# SEARLE: bộ chạy Colab của nhóm

Mỗi phiên Colab: chọn GPU (*Thời gian chạy → Thay đổi loại thời gian chạy*), rồi chạy lần lượt **ô 0 → ô 5**.
Sau đó dán lệnh của nhiệm vụ vào **mẫu ô lệnh** ở cuối notebook.

- Lần đầu: *Tệp → Lưu bản sao vào Drive*, rồi sửa `RUNNER` ở ô 0 thành tên của mình.
- Kế hoạch và lệnh từng nhiệm vụ: `plans/nhiem_vu_Chi.md` (đánh giá, thí nghiệm) và `plans/nhiem_vu_train_Colab.md` (train) trong repo.
- Kết quả mỗi lượt chạy có `--output-dir` nằm ở `$DRIVE/results/<mã>/`; nhật ký tự động ở `$DRIVE/results/runs_<RUNNER>.csv`.

### Ô 0. Cấu hình

In [2]:
RUNNER = "Chi"
DATASETS = "cirr circo fiq imagenet"
BRANCH = "main"

### Ô 1. Gắn Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.isdir('/content/drive/MyDrive/SEARLE'), \
    "Chưa thấy MyDrive/SEARLE: mở link thư mục SEARLE → chuột phải → Sắp xếp → Thêm lối tắt → Drive của tôi"
print("Drive OK")

Mounted at /content/drive
Drive OK


In [2]:
%%bash
# KHỞI ĐỘNG LẠI sau khi Colab ngắt: cài script hàng đợi + code mới (C17) lên Drive, rồi chạy boot.sh ở nền
D=/content/drive/MyDrive/SEARLE
mkdir -p $D/queue $D/code_changes
[ -f /content/queue_scripts.tar ] && tar xf /content/queue_scripts.tar -C $D/queue && echo "queue scripts installed"
[ -f /content/code_c17.tar ] && tar xf /content/code_c17.tar -C $D/code_changes && echo "code C17 saved to Drive"
ls $D/queue $D/code_changes
pgrep -f "queue/boot.sh" >/dev/null && echo "boot already running" || { nohup bash $D/queue/boot.sh > /content/boot.log 2>&1 & echo "boot started"; }

/content/drive/MyDrive/SEARLE/code_changes:
restore_oracle.json
restore_rules.json
SEARLE-main

/content/drive/MyDrive/SEARLE/queue:
boot.sh
common.sh
q11.log
q11.sh
q1.log
q1.sh
q2.log
q2.sh
q3.log
q3.sh
q4.log
q4.sh
q5.log
q5.sh
q6.log
q6.sh
q7.log
q7.sh
q8.log
q8.sh
q9.log
q9.sh
t1.log
t1.sh
t2.log
t2.sh
t3.log
t3.sh
boot started


In [60]:
# GIỮ KẾT NỐI + THEO DÕI: ô này chạy liên tục (Colab không coi là "rảnh" nên không tự ngắt).
# Mỗi 5 phút in trạng thái mới nhất (thay cho lần in trước). Muốn chạy ô khác: Thời gian chạy -> Tạm ngừng thực thi, xong thì chạy lại ô này.
import subprocess, time
from IPython.display import clear_output
D = '/content/drive/MyDrive/SEARLE'
cmd = f"""date; echo '== boot'; tail -n 3 /content/boot.log 2>/dev/null
for s in q11; do echo "== $s"; tail -n 3 {D}/queue/$s.log 2>/dev/null; done
grep -h FAIL {D}/queue/q*.log 2>/dev/null | tail -n 5
for f in {D}/results/E1.14a-train-s0/train.log {D}/results/E3.8a-train/*.log {D}/results/E3.8b-train-robust/train.log {D}/results/E8.2a-train-cosine/train.log; do [ -f $f ] && echo "$(basename $(dirname $f))/$(basename $f): $(tail -c 300 $f | tr '\\r' '\\n' | grep -v '^$' | tail -n 1 | cut -c1-90)"; done
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; df -h /content | tail -1"""
for i in range(24 * 12):
    try:
        out = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True, encoding='utf-8', errors='replace').stdout
    except Exception as e:
        out = f'monitor error: {e}'
    clear_output(wait=True)
    print(out, flush=True)
    time.sleep(300)

Sun Oct 25 10:22:01 AM UTC 2026
== boot
t2 started
q4 started
[02:34] boot done
== q11
xl_circo.json
xl_cirr.json
===== queue 11 done Sun Oct 25 10:15:28 AM UTC 2026
[12:29] FAIL  E4.3-flip-gallery-cirr:     raise ValueError(f"Experiment {args.exp_name} not found") ValueError: Experiment phi_B32 not found 
[12:29] FAIL  E4.3-flip-both-cirr:     raise ValueError(f"Experiment {args.exp_name} not found") ValueError: Experiment phi_B32 not found 
[12:29] FAIL  E1.4-ci-cirr: WARNING: skipped /content/drive/MyDrive/SEARLE/results/E1.9-*-cirr (no dump.jsonl) Baseline /content/drive/MyDrive/SEARLE/results/E1.7d-phiB32-cirr has no usable dump.jsonl 
[12:29] FAIL  E1.4-ci-circo: WARNING: skipped /content/drive/MyDrive/SEARLE/results/E1.8-*-circo-ex (no dump.jsonl) Baseline /content/drive/MyDrive/SEARLE/results/E1.8-phiB32-circo-ex has no usable dump.jsonl 
[12:29] FAIL  E1.4-ci-fiq: WARNING: skipped /content/drive/MyDrive/SEARLE/results/E1.9-*-fiq-ex (no dump.jsonl) Baseline /content/drive/MyDri

KeyboardInterrupt: 

In [61]:
# Kết quả mới: q5/q6 log, E3.6, E3.1 CIRCO-ex, E3.3, E2.3 trọng số lớn, train
import json, os, glob, subprocess
D = '/content/drive/MyDrive/SEARLE'; R = D + '/results'
print(subprocess.run(['bash','-c', f'tail -n 6 {D}/queue/q5.log; echo; tail -n 6 {D}/queue/q6.log; echo; tail -n 2 {D}/queue/t3.log'], capture_output=True, text=True, errors='replace').stdout)
def met(run):
    f = f'{R}/{run}/metrics.json'
    if not os.path.exists(f): return None
    m = json.load(open(f)); return m.get('metrics', m)
for d in sorted(glob.glob(R + '/E3.1-*-circo-ex')) + sorted(glob.glob(R + '/E3.3-*')) + sorted(glob.glob(R + '/E2.3-*-w[12].0-*')) + sorted(glob.glob(R + '/E2.3-*-w0.5-*')):
    x = met(os.path.basename(d))
    if x: print(f"{os.path.basename(d):34s} " + ' '.join(f"{k.replace('cirr_','').replace('circo_','').replace('average_fiq_','fiq_')}={v:.2f}" for k, v in x.items() if k in ('cirr_recall_at1','cirr_recall_at5','cirr_group_recall_at1','circo_map_at10','average_fiq_recall_at10','average_fiq_recall_at50')))
for d in sorted(glob.glob(R + '/E3.6-*')):
    f = glob.glob(d + '/*.md')
    if f:
        t = open(f[0]).read(); i = t.find('| all |')
        print(os.path.basename(d), t[i:i+70].replace(chr(10),' ') if i >= 0 else '?')
for p in ['phi_B32_robust', 'phi_B32_cosine']:
    print(p, open(f'{D}/outputs/phi_models/{p}/validation_metrics.csv').read().strip().splitlines()[-1][:45])

[21:34] OK    E2.7-ci-cirr
[21:34] START E2.7-ci-fiq
[21:34] OK    E2.7-ci-fiq
[21:34] START E2.7-ci-circo
[21:34] OK    E2.7-ci-circo
===== queue 5 done Sat Oct 24 09:34:55 PM UTC 2026

[19:40] OK    E3.3-mixed-auto-circo-ex
[19:40] START E3.3-mixed-oracle-circo-ex
[19:41] OK    E3.3-mixed-oracle-circo-ex
[19:41] START E3.3-ci-cirr
[19:41] OK    E3.3-ci-cirr
===== queue 6 done Sat Oct 24 07:41:30 PM UTC 2026

[16:02] START phi_B32_cosine 
[21:46] END   phi_B32_cosine (OK)

E3.1-color_cast1-circo-ex          map_at10=9.85
E3.1-color_cast2-circo-ex          map_at10=10.32
E3.1-gauss_blur1-circo-ex          map_at10=9.13
E3.1-gauss_blur2-circo-ex          map_at10=9.48
E3.1-gauss_noise1-circo-ex         map_at10=8.79
E3.1-gauss_noise2-circo-ex         map_at10=8.88
E3.1-jpeg1-circo-ex                map_at10=9.22
E3.1-jpeg2-circo-ex                map_at10=9.38
E3.1-low_light1-circo-ex           map_at10=8.70
E3.1-low_light2-circo-ex           map_at10=7.28
E3.1-low_res1-circo-ex        

In [22]:
%%bash
# Hàng đợi 7: màu và độ sáng với trọng số rất lớn (R@1 vẫn tăng tới w = 2), tìm đỉnh
source /content/env.sh; Q=$DRIVE/queue
cat > $Q/q7.sh <<'EOF'
#!/bin/bash
source $DRIVE/queue/common.sh
echo "===== queue 7 start $(date)"
for FAM in color brightness; do
  for W in 4.0 8.0; do
    run $R/E2.3-$FAM-w$W-cirr $V $PHI $CIRR --attr-rerank --attr-families $FAM --attr-weight $W --output-dir $R/E2.3-$FAM-w$W-cirr
    run $R/E2.3-$FAM-w$W-fiq  $V $PHI $FIQ --exclude-reference --attr-rerank --attr-families $FAM --attr-weight $W --output-dir $R/E2.3-$FAM-w$W-fiq
  done
done
run $R/E2.3-color-w2.0-k50-cirr $V $PHI $CIRR --attr-rerank --attr-families color --attr-weight 2.0 --attr-topk 50 --output-dir $R/E2.3-color-w2.0-k50-cirr
run $R/E2.3-color-w2.0-k200-cirr $V $PHI $CIRR --attr-rerank --attr-families color --attr-weight 2.0 --attr-topk 200 --output-dir $R/E2.3-color-w2.0-k200-cirr
echo "===== queue 7 done $(date)"
EOF
bash -n $Q/q7.sh && { nohup bash $Q/q7.sh >> $Q/q7.log 2>&1 & echo q7 started; }

q7 started


In [18]:
# Bảng E2.3 (họ thuộc tính × trọng số), E4.4, E4.5, E1.10 taxonomy
import json, os, glob
R = '/content/drive/MyDrive/SEARLE/results'
def met(run):
    f = f'{R}/{run}/metrics.json'
    if not os.path.exists(f): return None
    m = json.load(open(f)); return m.get('metrics', m)
b_c = met('E1.7d-phiB32-cirr'); b_f = met('E1.8-phiB32-fiq-ex')
print('base cirr R1 %.2f R5 %.2f Rs1 %.2f | fiq R10 %.2f R50 %.2f' % (b_c['cirr_recall_at1'], b_c['cirr_recall_at5'], b_c['cirr_group_recall_at1'], b_f['average_fiq_recall_at10'], b_f['average_fiq_recall_at50']))
for F in ['color', 'brightness', 'colorfulness', 'pattern', 'zoom']:
    s = F[:6]
    for W in ['0.05', '0.1', '0.2', '0.3']:
        c = met(f'E2.3-{F}-w{W}-cirr'); f = met(f'E2.3-{F}-w{W}-fiq')
        s += f" | w{W} " + (f"{c['cirr_recall_at1']:.2f}/{c['cirr_recall_at5']:.2f}" if c else '-') + ' ' + (f"{f['average_fiq_recall_at10']:.2f}/{f['average_fiq_recall_at50']:.2f}" if f else '-')
    print(s)
for r in ['E4.4-fusion-a0.05-cirr','E4.4-fusion-a0.1-cirr','E4.4-fusion-a0.2-cirr','E4.4-fusion-a0.3-cirr','E4.4-fusion-a0.5-cirr','E4.5-ensemble-cirr','E4.1-pad1.0-cirr','E4.1-pad1.5-cirr','E4.1-pad2.0-cirr','E4.1-clipcrop-cirr','E4.2-interp-nearest-cirr','E4.2-interp-bilinear-cirr','E4.2-interp-lanczos-cirr','E4.3-flip-ref-cirr','E4.3-flip-gallery-cirr','E4.3-flip-both-cirr','E3.5-lp0.2-cirr','E3.5-lp0.3-cirr','E3.5-lp0.5-cirr','E3.5-hp0.1-cirr','E3.5-hp0.2-cirr','E3.5-hp0.3-cirr','E3.5-hp0.5-cirr','E1.14-phi_scale_s0-cirr','E1.14-phi_scale_s0-circo']:
    x = met(r)
    if x: print(r, ' '.join(f"{k.replace('cirr_','').replace('circo_','')}={v:.2f}" for k, v in x.items() if k in ('cirr_recall_at1','cirr_recall_at5','cirr_group_recall_at1','circo_map_at10')))
f = f'{R}/E3.8a-choice.txt'
print(open(f).read() if os.path.exists(f) else 'no choice file')

base cirr R1 25.47 R5 55.23 Rs1 55.20 | fiq R10 22.74 R50 41.57
color | w0.05 25.45/55.25 22.76/41.64 | w0.1 25.45/55.27 22.79/41.66 | w0.2 25.47/55.37 22.83/41.71 | w0.3 25.52/55.44 22.88/41.72
bright | w0.05 25.47/55.25 22.74/41.57 | w0.1 25.47/55.27 22.76/41.57 | w0.2 25.47/55.25 22.78/41.71 | w0.3 25.47/55.27 22.78/41.67
colorf | w0.05 25.47/55.23 22.74/41.57 | w0.1 25.47/55.23 22.74/41.57 | w0.2 25.47/55.18 22.73/41.57 | w0.3 25.47/55.18 22.73/41.57
patter | w0.05 25.47/55.23 22.76/41.57 | w0.1 25.47/55.23 22.78/41.56 | w0.2 25.47/55.23 22.81/41.59 | w0.3 25.47/55.23 22.79/41.61
zoom | w0.05 25.50/55.27 22.74/41.57 | w0.1 25.47/55.25 22.74/41.57 | w0.2 25.50/55.25 22.74/41.57 | w0.3 25.45/55.25 22.74/41.57
E4.4-fusion-a0.05-cirr recall_at1=23.99 recall_at5=54.72 group_recall_at1=52.45
E4.4-fusion-a0.1-cirr recall_at1=22.82 recall_at5=53.96 group_recall_at1=49.82
E4.4-fusion-a0.2-cirr recall_at1=20.50 recall_at5=51.28 group_recall_at1=45.64
E4.4-fusion-a0.3-cirr recall_at1=18.27 re

In [24]:
%%bash
echo aW1wb3J0IGpzb24KUj0nL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9TRUFSTEUvcmVzdWx0cy9FMTAuMC1zdGF0cy0nCmZvciBuIGluIFsnR1BUTmVvMjdCJywnUDJfcXdlbjE1X3ZhbCcsJ1AzX3F3ZW4xNWlfdmFsJ106CiAgaj1qc29uLmxvYWQob3BlbihSK24rJy9zdGF0cy5qc29uJykpCiAgZGVmIGZsKHgscD0nJyk6CiAgICBvPXt9CiAgICBmb3Igayx2IGluIHguaXRlbXMoKToKICAgICAgaWYgaXNpbnN0YW5jZSh2LGRpY3QpOiBvLnVwZGF0ZShmbCh2LHAraysnLicpKQogICAgICBlbGlmIGlzaW5zdGFuY2UodiwoaW50LGZsb2F0KSk6IG9bcCtrXT1yb3VuZCh2LDMpCiAgICByZXR1cm4gbwogIHByaW50KG4sIGpzb24uZHVtcHMoZmwoaikpWzo5MDBdKQ== | base64 -d | /usr/bin/python3
D=/content/drive/MyDrive/SEARLE; /usr/bin/python3 -c "import pickle;d=pickle.load(open('$D/outputs/GPT_phrases/P3_qwen15i_val/concept_to_phrases.pkl','rb'));ks=list(d);import random;random.seed(1);[print(k,'->',d[k][:2]) for k in random.sample(ks,3)]"

GPTNeo27B {"concepts": 20931, "phrases": 5358336, "phrases_per_concept.min": 256, "phrases_per_concept.mean": 256.0, "phrases_per_concept.max": 256, "concepts_without_phrases": 0, "duplicate_phrases_pct": 0.06, "concepts_dropped_by_original_filter": 2, "concepts_dropped_by_original_filter_pct": 0.01, "phrases_with_concept_several_times_pct": 12.13, "phrases_with_concept_only_inside_a_word_pct": 0.0, "phrases_with_repeated_words_pct": 9.43, "phrases_with_non_ascii_pct": 14.09, "words_per_phrase.p5": 22.0, "words_per_phrase.median": 28.0, "words_per_phrase.p95": 32.0, "clip_tokens_sample.n": 26900, "clip_tokens_sample.median": 33.0, "clip_tokens_sample.over_75_pct": 0.0, "distinct2_mean": 0.365, "used_concepts.images": 2165, "used_concepts.oti_top_k": 15, "used_concepts.phi_top_k": 0, "used_concepts.oti_concepts": 6492, "used_concepts.phi_concepts": 0, "used_concepts.union": 6492, "used_concepts.uni
P2_qwen15_val {"concepts": 7514, "phrases": 472264, "phrases_per_concept.min": 30, "phras

In [15]:
# E3.2 -> restore_oracle.json (phép phục hồi tốt nhất mỗi loại suy giảm, theo CIRR R@1), cài hàng đợi 6 (E3.3), xem train
import json, os, glob, subprocess
D = '/content/drive/MyDrive/SEARLE'; R = D + '/results'; SRC = '/content/XLA_SEARLE/SEARLE-main'
def met(run):
    f = f'{R}/{run}/metrics.json'
    if not os.path.exists(f): return None
    m = json.load(open(f)); return m.get('metrics', m)
rows, best = [], {}
for kind in ['gauss_noise', 'salt_pepper', 'gauss_blur', 'motion_blur', 'jpeg', 'low_light', 'low_res', 'color_cast']:
    base = met(f'E3.1-{kind}2-cirr')
    cands = {'none': base['cirr_recall_at1'] if base else None}
    for f in sorted(glob.glob(f'{R}/E3.2-{kind}2-*-cirr')):
        meth = os.path.basename(f)[len(f'E3.2-{kind}2-'):-len('-cirr')]
        mm = met(os.path.basename(f))
        if mm: cands[meth] = mm['cirr_recall_at1']; rows.append((kind, meth, mm['cirr_recall_at1'], mm.get('psnr_mean') or mm.get('restored_psnr')))
    c = {k: v for k, v in cands.items() if v is not None}
    best[kind] = max(c, key=c.get) if c else 'none'
    print(f"{kind:12s} " + '  '.join(f"{k}={v:.2f}" for k, v in c.items()) + f"   -> {best[kind]}")
oracle = {'note': 'Best restoration per degradation for --restore oracle: best CIRR val R@1 at level 2 in E3.2 (none = E3.1, no restoration).', 'source': 'E3.2 (24/10/2026, A100, phi_B32)', 'methods': best}
for p in [SRC + '/src/restore_oracle.json', D + '/code_changes/restore_oracle.json']:
    json.dump(oracle, open(p, 'w'), indent=2)
print(subprocess.run(['bash', '-c', 'source /content/env.sh; Q=$DRIVE/queue; tar xf /content/fix2.tar -C $Q; pgrep -f queue/q6.sh >/dev/null && echo q6 running || { nohup bash $Q/q6.sh >> $Q/q6.log 2>&1 & echo q6 started; }; for p in phi_B32_robust phi_B32_cosine; do echo "$p: $(tail -n 1 $SRC/data/phi_models/$p/validation_metrics.csv | cut -c1-40)"; done; grep -c OK $Q/q2.log; tail -n 2 $Q/q2.log'], capture_output=True, text=True, errors='replace').stdout)

gauss_noise  none=24.49  bilateral=24.61  gaussian=24.35  nlm=23.80   -> bilateral
salt_pepper  none=23.39  median=25.16   -> median
gauss_blur   none=25.07  richardson_lucy=24.95  unsharp=25.21  wiener=24.92   -> unsharp
motion_blur  none=22.70  richardson_lucy=24.28  wiener=24.66   -> wiener
jpeg         none=23.89  bilateral=24.30  deblock=24.13   -> bilateral
low_light    none=23.22  clahe=23.51  gamma=23.97  histeq=23.06  retinex=23.85   -> gamma
low_res      none=25.04  unsharp=25.19   -> unsharp
color_cast   none=24.78  gray_world=24.87  white_patch=24.85   -> gray_world
q6 running
phi_B32_robust: 30,24.396077493422627,54.72375029897154,
phi_B32_cosine: 40,24.34824204735709,54.14972494618512,6
55
[18:24] OK    E2.3-zoom-w0.1-cirr
[18:24] START E2.3-zoom-w0.1-fiq



In [16]:
%%bash
source /content/env.sh; Q=$DRIVE/queue
pgrep -f "queue/q6[.]sh" >/dev/null && echo "q6 already running" || { nohup bash $Q/q6.sh >> $Q/q6.log 2>&1 & echo "q6 started"; }
sleep 3; ps aux | grep -c "[q]ueue/q6.sh"; cat $Q/q6.log 2>/dev/null | tail -3

q6 started
1


In [12]:
%%bash
# Cho hàng đợi 3 chạy song song hàng đợi 2; H16 cosine chạy ngay (t3)
Q=/content/drive/MyDrive/SEARLE/queue
pkill -f "queue/q3.sh"; pkill -f "queue/t2.sh"; pkill -f "queue/q4.sh"; sleep 1
ls /content/fix1.tar && tar xf /content/fix1.tar -C $Q
for s in t3 q3 t2 q4; do pgrep -f "queue/$s.sh" >/dev/null && echo "$s already running" || { nohup bash $Q/$s.sh >> $Q/$s.log 2>&1 & echo "$s started"; sleep 2; }; done
sleep 20; tail -n 3 $Q/t3.log $Q/q3.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

/content/fix1.tar
t3 started
q3 started
t2 started
q4 started
==> /content/drive/MyDrive/SEARLE/queue/t3.log <==
===== train queue 3 start Sat Oct 24 04:01:48 PM UTC 2026
[16:01] START phi_B32_cosine 
[16:01] END   phi_B32_cosine (FAIL)

==> /content/drive/MyDrive/SEARLE/queue/q3.log <==
/content/drive/MyDrive/SEARLE/queue/q3.sh: line 4: /queue/common.sh: No such file or directory
0 %, 2706 MiB


In [13]:
%%bash
source /content/env.sh; Q=$DRIVE/queue
pkill -f "queue/q3.sh"; pkill -f "queue/t2.sh"; pkill -f "queue/q4.sh"; pkill -f "queue/t3.sh"; sleep 1
for s in t3 q3 t2 q4; do nohup bash $Q/$s.sh >> $Q/$s.log 2>&1 & echo "$s started"; sleep 2; done
sleep 45; tail -n 3 $Q/t3.log $Q/q3.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

t3 started
q3 started
t2 started
q4 started
==> /content/drive/MyDrive/SEARLE/queue/t3.log <==
[16:01] END   phi_B32_cosine (FAIL)
===== train queue 3 start Sat Oct 24 04:02:40 PM UTC 2026
[16:02] START phi_B32_cosine 

==> /content/drive/MyDrive/SEARLE/queue/q3.log <==
[16:02] START E3.8a-phi_s0_clean-none-cirr
[16:03] OK    E3.8a-phi_s0_clean-none-cirr
[16:03] START E3.8a-phi_s0_clean-gauss_noise2-cirr
6 %, 7172 MiB


In [9]:
%%bash
# Hàng đợi 5: độ bền CIRCO không có ảnh tham chiếu, các lựa chọn E2.4-E2.7
Q=/content/drive/MyDrive/SEARLE/queue
tar xf /content/q5_install.tar -C $Q && ls -la $Q/q5.sh $Q/boot.sh
pgrep -f "queue/q5.sh" >/dev/null && echo "q5 already running" || { nohup bash $Q/q5.sh >> $Q/q5.log 2>&1 & echo "q5 started"; }

-rw------- 1 root root 2008 Oct 24 14:10 /content/drive/MyDrive/SEARLE/queue/boot.sh
-rw------- 1 root root 5597 Oct 24 14:09 /content/drive/MyDrive/SEARLE/queue/q5.sh
q5 started


In [7]:
# TIẾN ĐỘ: lượt xong / lỗi, số chính của các lượt mới
import json, glob, os
R = '/content/drive/MyDrive/SEARLE/results'
dirs = sorted(d for d in glob.glob(R + '/*') if os.path.isdir(d))
done = [os.path.basename(d) for d in dirs if os.path.exists(d + '/.done')]
notdone = [os.path.basename(d) for d in dirs if os.path.exists(d + '/stdout.log') and not os.path.exists(d + '/.done')]
print(len(done), 'done |', len(notdone), 'started but not done:', ' '.join(notdone))
KEYS = ['cirr_recall_at1', 'circo_map_at10', 'fiq_recall_at10', 'fiq_recall_at50']
for d in done:
    f = f'{R}/{d}/metrics.json'
    if not os.path.exists(f) or d.startswith(('E1.7', 'E1.8')): continue
    mt = json.load(open(f)); mt = mt.get('metrics', mt)
    s = ' '.join(f"{k.split('_',1)[1]}={mt[k]:.2f}" for k in mt if any(k.endswith(x.split('_',1)[1]) and k.startswith(x.split('_')[0]) for x in KEYS) and isinstance(mt[k], (int, float)))
    print(f'{d:34s} {s}')
for p in ['phi_scale_s0', 'phi_s0_clean', 'phi_s0_degaug', 'phi_s0_noise']:
    f = f'/content/drive/MyDrive/SEARLE/outputs/phi_models/{p}/validation_metrics.csv'
    if os.path.exists(f): print(p, 'last val:', open(f).read().strip().splitlines()[-1][:60])

70 done | 1 started but not done: E3.5-hp0.1-cirr
E1.9-cbir-all-cirr                 recall_at1=1.72 group_recall_at1=20.23
E1.9-cbir-all-fiq-ex               
E1.9-cbir-color-cirr               recall_at1=1.22 group_recall_at1=19.71
E1.9-cbir-color-fiq-ex             
E1.9-cbir-gabor-cirr               recall_at1=0.38 group_recall_at1=20.11
E1.9-cbir-gabor-fiq-ex             
E1.9-cbir-hog-cirr                 recall_at1=1.15 group_recall_at1=19.95
E1.9-cbir-hog-fiq-ex               
E1.9-cbir-lbp-cirr                 recall_at1=1.08 group_recall_at1=20.78
E1.9-cbir-lbp-fiq-ex               
E1.9-cbir-text-cirr                recall_at1=11.58 group_recall_at1=41.35
E3.1-color_cast1-circo             map_at10=7.91
E3.1-color_cast1-cirr              recall_at1=24.95 group_recall_at1=55.39
E3.1-color_cast2-circo             map_at10=8.50
E3.1-color_cast2-cirr              recall_at1=24.78 group_recall_at1=55.68
E3.1-gauss_blur1-circo             map_at10=7.09
E3.1-gauss_blur1-cirr       

### Ô 2. Lấy code mới nhất từ GitHub

In [6]:
import os
REPO = "/content/XLA_SEARLE"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} remote set-url origin https://github.com/chilk1307/Chiss.git
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/chilk1307/Chiss.git {REPO}
!git -C {REPO} log -1 --format="Code: %h  %s  (%cr)"

Code: 924d9c4  Chí làm một mình: gộp nhiệm vụ, thêm kế hoạch train Colab, dữ liệu dùng thư mục giải nén  (13 minutes ago)


### Ô 3. Cài môi trường (Python 3.8, vài phút ở lần đầu mỗi phiên)

In [7]:
!bash /content/XLA_SEARLE/SEARLE-main/colab/setup_env.sh {RUNNER}

Creating the Python 3.8 venv (a few minutes)...
Using CPython 3.8.20
Creating virtual environment at: venv
Activate with: source venv/bin/activate
python 3.8.20 | torch 1.11.0+cu113 | torchvision 0.12.0+cu113 | numpy 1.23.5 | opencv 4.10.0 | scikit-image 0.21.0 | scipy 1.10.1
CUDA: True NVIDIA A100-SXM4-40GB
Wrote /content/env.sh (RUNNER=Chi)


### Ô 4. Chuẩn bị dữ liệu (giải nén từ Drive ra `/content/data`)

In [8]:
!bash /content/XLA_SEARLE/SEARLE-main/colab/prepare_data.sh {DATASETS}

CIRR: linking /content/data/CIRR to the folder /content/drive/MyDrive/SEARLE/datasets/CIRR
CIRR: ready in /content/data/CIRR
CIRCO: linking /content/data/CIRCO to the folder /content/drive/MyDrive/SEARLE/datasets/CIRCO
CIRCO: ready in /content/data/CIRCO
FashionIQ: linking /content/data/FashionIQ to the folder /content/drive/MyDrive/SEARLE/datasets/FashionIQ
FashionIQ: ready in /content/data/FashionIQ (29197 images)
ImageNet1K: extracting /content/drive/MyDrive/SEARLE/datasets/ImageNet1K/ILSVRC2012_img_test_v10102019.tar to the local disk (about 13 GB, several minutes) ...
ImageNet1K: ready (100000 test images)
Linked /content/XLA_SEARLE/SEARLE-main/data/{GPT_phrases,oti_pseudo_tokens,similar_concepts,phi_models} -> /content/drive/MyDrive/SEARLE/outputs/
Local disk: 164G free of 236G


### Ô 5. Kiểm tra GPU

In [9]:
%%bash
source /content/env.sh
nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Không có GPU: Thời gian chạy → Thay đổi loại thời gian chạy"
$PY -c "import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
torch 1.11.0+cu113 | CUDA True


---
## Kiểm thử bộ chạy (C2, chạy một lần khi có CIRR)

Kỳ vọng: `cirr_recall_at1 = 25.66` (±0.05). Lần đầu tải trọng số SEARLE qua `torch.hub` (cần internet).

In [ ]:
%%bash
source /content/env.sh
cd $SRC
time $PY src/validate.py --eval-type searle --dataset cirr --dataset-path $DATA/CIRR

## Kiểm tra dữ liệu (E1.2)

Chạy cho các tập đã chuẩn bị ở ô 4; kết quả ở `$DRIVE/results/E1.2-dataset-check/report.md`.

In [19]:
%%bash
source /content/env.sh
cd $SRC
for DS in cirr circo fashioniq; do
  P=$DATA/CIRR; [ $DS = circo ] && P=$DATA/CIRCO; [ $DS = fashioniq ] && P=$DATA/FashionIQ
  if [ -d $P ]; then $PY tools/check_datasets.py --dataset $DS --dataset-path $P --out $DRIVE/results/E1.2-dataset-check; fi
done

## cirr

Dataset path: `/content/data/CIRR`

### train

| | |
|---|---|
| images_declared | 16939 |
| images_present | 16939 |
| images_missing_examples | [] |
| queries | 28225 |
| queries_reference_missing | 0 |
| queries_target_missing | 0 |
| queries_with_group_member_missing | 0 |

### val

| | |
|---|---|
| images_declared | 2297 |
| images_present | 2297 |
| images_missing_examples | [] |
| queries | 4181 |
| queries_reference_missing | 0 |
| queries_target_missing | 0 |
| queries_with_group_member_missing | 0 |

### test1

| | |
|---|---|
| images_declared | 2315 |
| images_present | 2315 |
| images_missing_examples | [] |
| queries | 4148 |
| queries_reference_missing | 0 |
| queries_target_missing | None |
| queries_with_group_member_missing | 0 |


Written /content/drive/MyDrive/SEARLE/results/E1.2-dataset-check/report.md
## fashioniq

Dataset path: `/content/data/FashionIQ`

### train/dress

| | |
|---|---|
| queries | 5985 |
| queries_with_all_images | 5440 |
| gallery_dec

Traceback (most recent call last):
  File "tools/check_datasets.py", line 142, in <module>
    main()
  File "tools/check_datasets.py", line 126, in main
    result = check_circo(path)
  File "tools/check_datasets.py", line 46, in check_circo
    on_disk = existing_files(path / 'COCO2017_unlabeled' / 'unlabeled2017')
  File "tools/check_datasets.py", line 18, in existing_files
    return set(os.listdir(folder)) if folder.is_dir() else set()
OSError: [Errno 5] Input/output error: '/content/data/CIRCO/COCO2017_unlabeled/unlabeled2017'


In [20]:
%%bash
# Bù ảnh FashionIQ val còn thiếu từ bản sao Hugging Face royokong/fashioniq_val_imgs (15.536 ảnh val)
set -e
mkdir -p /content/fiq_hf && cd /content/fiq_hf
[ -s val.parquet ] || wget -q -O val.parquet https://huggingface.co/datasets/royokong/fashioniq_val_imgs/resolve/main/data/val-00000-of-00001.parquet
ls -la val.parquet
python3 - <<'EOF'
import pandas as pd, os, io, json
from PIL import Image
df = pd.read_parquet('/content/fiq_hf/val.parquet')
print(df.shape, df[['category', 'split', 'id']].head(3).to_dict('records'))
dst = '/content/data/FashionIQ/images'
have = {f[:-4] for f in os.listdir(dst) if f.endswith('.jpg')}
ids = set(df['id'])
declared = set()
for c in ['dress', 'shirt', 'toptee']:
    declared |= set(json.load(open(f'/content/data/FashionIQ/image_splits/split.{c}.val.json')))
print('val declared', len(declared), '| in parquet', len(declared & ids), '| already on Drive', len(declared & have))
assert len(ids & have) > 1000, 'id format does not match file names, stop'
added = 0
for r in df.itertuples():
    if r.id in have:
        continue
    b = r.img['bytes']
    path = os.path.join(dst, r.id + '.jpg')
    im = Image.open(io.BytesIO(b))
    if im.format == 'JPEG':
        open(path, 'wb').write(b)
    else:
        im.convert('RGB').save(path, quality=95)
    added += 1
    if added % 1000 == 0:
        print('added', added, flush=True)
print('added', added, '| val still missing', len(declared - have - ids))
EOF

-rw-r--r-- 1 root root 207236534 Oct 24 09:54 val.parquet
(15536, 4) [{'category': 'dress', 'split': 'val', 'id': 'B009PMCJLW'}, {'category': 'dress', 'split': 'val', 'id': 'B0084Y8XIU'}, {'category': 'dress', 'split': 'val', 'id': 'B007HNF4QI'}]
val declared 15415 | in parquet 15415 | already on Drive 6079
added 1000
added 2000
added 3000
added 4000
added 5000
added 6000
added 7000
added 8000
added 9000
added 9418 | val still missing 0


In [21]:
%%bash
source /content/env.sh
cd $SRC
$PY tools/check_datasets.py --dataset fashioniq --dataset-path $DATA/FashionIQ --out $DRIVE/results/E1.2-dataset-check | grep -v "^| | |" | grep -v "^|---"

## fashioniq

Dataset path: `/content/data/FashionIQ`

### train/dress

| queries | 5985 |
| queries_with_all_images | 5440 |
| gallery_declared | 11452 |
| gallery_present | 10874 |

### train/shirt

| queries | 5988 |
| queries_with_all_images | 760 |
| gallery_declared | 19036 |
| gallery_present | 5400 |

### train/toptee

| queries | 6027 |
| queries_with_all_images | 352 |
| gallery_declared | 16121 |
| gallery_present | 2459 |

### val/dress

| queries | 2017 |
| queries_with_all_images | 2017 |
| gallery_declared | 3817 |
| gallery_present | 3817 |

### val/shirt

| queries | 2038 |
| queries_with_all_images | 2038 |
| gallery_declared | 6346 |
| gallery_present | 6346 |

### val/toptee

| queries | 1961 |
| queries_with_all_images | 1961 |
| gallery_declared | 5373 |
| gallery_present | 5373 |

### test/dress

| queries | 2024 |
| queries_with_all_images | 1904 |
| gallery_declared | 3818 |
| gallery_present | 3604 |

### test/shirt

| queries | 2038 |
| queries_with_all_image

In [22]:
%%bash
source /content/env.sh
echo "--- T1"; tail -c 600 $DRIVE/results/E1.14a-train-s0/train.log
echo; echo "--- D7"; tail -c 400 $DRIVE/results/E3.8a-train/precompute_s0.log
echo; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

--- T1
[15/100] total_loss: 4.825 gpt_loss: 0.143 distil_loss: 4.718:  19%|█████████▍                                       | 25/130 [00:09<00:35,  2.92it/s]
--- D7
part 7/7: 100%|██████████| 105/105 [01:38<00:00,  1.07it/s]
Wrote 9 feature files of 33334 images to /content/drive/MyDrive/SEARLE/cache/imagenet_s0

utilization.gpu [%], memory.used [MiB]
67 %, 16252 MiB


In [23]:
%%bash
# CIRCO: Drive không liệt kê nổi 123k ảnh (lỗi I/O), nên tải ảnh COCO unlabeled2017 thẳng từ cocodataset.org ra đĩa Colab
source /content/env.sh
set -e
D=$DRIVE/datasets/CIRCO
L=/content/data/CIRCO_local
echo "Drive CIRCO:"; ls $D $D/COCO2017_unlabeled $D/annotations
echo "Drive unlabeled2017 count (có thể lỗi I/O): $(ls -f $D/COCO2017_unlabeled/unlabeled2017 2>&1 | wc -l)"
mkdir -p $L/COCO2017_unlabeled
cp -rn $D/annotations $L/
cp -rn $D/COCO2017_unlabeled/annotations $L/COCO2017_unlabeled/
cd /content
[ -s unlabeled2017.zip ] || time wget -q -c http://images.cocodataset.org/zips/unlabeled2017.zip
ls -la unlabeled2017.zip
[ -d $L/COCO2017_unlabeled/unlabeled2017 ] || time unzip -q unlabeled2017.zip -d $L/COCO2017_unlabeled/
echo "local unlabeled2017: $(ls -f $L/COCO2017_unlabeled/unlabeled2017 | grep -c jpg) images"
cd $SRC && $PY tools/check_datasets.py --dataset circo --dataset-path $L --out $DRIVE/results/E1.2-dataset-check | grep -v "^| | |" | grep -v "^|---"
df -h /content | tail -1

Drive CIRCO:
/content/drive/MyDrive/SEARLE/datasets/CIRCO:
annotations
assets
COCO2017_unlabeled
LICENSE
README.md
src
submission_examples

/content/drive/MyDrive/SEARLE/datasets/CIRCO/annotations:
test.json
val.json

/content/drive/MyDrive/SEARLE/datasets/CIRCO/COCO2017_unlabeled:
annotations
image_info_unlabeled2017.zip
unlabeled2017
Drive unlabeled2017 count (có thể lỗi I/O): 123405
-rw-r--r-- 1 root root 20126613414 Jul 11  2018 unlabeled2017.zip
local unlabeled2017: 123403 images
## circo

Dataset path: `/content/data/CIRCO_local`

### gallery

| images_declared | 123403 |
| images_present | 123403 |
| images_missing | 0 |
| images_missing_examples | [] |

### val

| queries | 220 |
| queries_reference_missing | 0 |
| queries_target_missing | 0 |
| ground_truths | 916 |
| ground_truths_missing | 0 |

### test

| queries | 800 |
| queries_reference_missing | 0 |


Written /content/drive/MyDrive/SEARLE/results/E1.2-dataset-check/report.md
overlay         236G  117G  119G  50% /


cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead

real	5m31.653s
user	0m0.759s
sys	0m24.026s

real	2m12.430s
user	1m43.287s
sys	0m20.149s


In [24]:
%%bash
source /content/env.sh
OLD=/content/drive/MyDrive/outputs; NEW=$DRIVE/outputs
for d in GPT_phrases similar_concepts oti_pseudo_tokens phi_models/phi_B32; do
  o=$(cd $OLD/$d && find . -type f ! -name '*.gsheet' -printf '%p %s\n' | sort | md5sum | cut -c1-8)
  n=$(cd $NEW/$d && find . -type f ! -name '*.gsheet' -printf '%p %s\n' | sort | md5sum | cut -c1-8)
  echo "$d: files old=$(find $OLD/$d -type f ! -name '*.gsheet' | wc -l) new=$(find $NEW/$d -type f | wc -l) list+size $( [ $o = $n ] && echo KHOP || echo LECH )"
done
echo "ImageNet test (local): $(ls -f $DATA/ImageNet1K/test | grep -ci '\.jpe\?g$') images"
echo "FIQ val subset: $(python3 -c "import json;d=json.load(open('$DRIVE/results/E1.2-dataset-check/fiq_val_subset.json'));print({k:len(v) for k,v in d.items()})")"
echo "--- T1"; tail -c 300 $DRIVE/results/E1.14a-train-s0/train.log | tr '\r' '\n' | tail -2

GPT_phrases: files old=0 new=1 list+size LECH
similar_concepts: files old=0 new=1 list+size LECH
oti_pseudo_tokens: files old=0 new=18 list+size LECH
phi_models/phi_B32: files old=0 new=25 list+size LECH
ImageNet test (local): 100000 images
FIQ val subset: {'dress': 2017, 'shirt': 2038, 'toptee': 1961}
--- T1
��█████████▋              | 92/130 [00:28<00:11,  3.45it/s]
[28/100] total_loss: 4.787 gpt_loss: 0.142 distil_loss: 4.680:  72%|███████████████████████████████████              | 93/130 [00:28<00:10,  3.46it/s]

bash: line 4: cd: /content/drive/MyDrive/outputs/GPT_phrases: No such file or directory
find: ‘/content/drive/MyDrive/outputs/GPT_phrases’: No such file or directory
bash: line 6: [: =: unary operator expected
bash: line 4: cd: /content/drive/MyDrive/outputs/similar_concepts: No such file or directory
find: ‘/content/drive/MyDrive/outputs/similar_concepts’: No such file or directory
bash: line 6: [: =: unary operator expected
bash: line 4: cd: /content/drive/MyDrive/outputs/oti_pseudo_tokens: No such file or directory
find: ‘/content/drive/MyDrive/outputs/oti_pseudo_tokens’: No such file or directory
bash: line 6: [: =: unary operator expected
bash: line 4: cd: /content/drive/MyDrive/outputs/phi_models/phi_B32: No such file or directory
find: ‘/content/drive/MyDrive/outputs/phi_models/phi_B32’: No such file or directory
bash: line 6: [: =: unary operator expected


In [25]:
%%bash
source /content/env.sh
date
echo "--- T1"; tail -c 400 $DRIVE/results/E1.14a-train-s0/train.log | tr '\r' '\n' | grep -v '^$' | tail -2
echo; echo "--- validation T1"; tail -n 4 $SRC/data/phi_models/phi_scale_s0/validation_metrics.csv 2>/dev/null
ls $SRC/data/phi_models/phi_scale_s0/checkpoints 2>/dev/null | tr '\n' ' '
echo; ps aux | grep -E 'train_phi|precompute|validate' | grep -v grep | awk '{print $2, $10, $12, $13, $14, $15}'
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

Sat Oct 24 10:31:26 AM UTC 2026
--- T1
[54/100] total_loss: 4.756 gpt_loss: 0.142 distil_loss: 4.649:  28%|█████████████▉                                   | 37/130 [00:12<00:27,  3.43it/s]
[54/100] total_loss: 4.756 gpt_loss: 0.142 distil_loss: 4.649:  29%|██████████████▎                                  | 38/130 [00:12<00:27,  3.30it/s]

--- validation T1
35,23.965558478832815,53.240851470939965,66.96962449174838,89.21310691222195,55.41736426692179,76.70413776608467,87.92154986845252
40,23.989476201865582,53.647452762497004,67.04137766084669,89.06960057402536,55.345611097823486,76.84764410428127,88.08897392968188
45,24.08514709399665,53.59961731643148,67.20880172207606,89.11743602009089,55.36952882085625,76.6802200430519,87.89763214541976
50,24.324324324324326,53.67137048552978,67.01745993781391,89.14135374312366,55.44128198995456,76.7519732121502,87.89763214541976
phi_10.pt phi_15.pt phi_20.pt phi_25.pt phi_30.pt phi_35.pt phi_40.pt phi_45.pt phi_50.pt phi_5.pt phi_last.pt resume.pt

In [26]:
%%bash
# Hàng đợi chạy nền: q1 = các lượt đánh giá (H5, H4, H9, H6, D5, D6, D3, D4, H7), t1 = train sàng lọc phi bền (D8) + đặc trưng 100k (D9)
source /content/env.sh
Q=$DRIVE/queue; mkdir -p $Q
echo 'H4sIAAAAAAAAA+1ae2/bRhL33/wUcwyTSK0pivIrkeOgih+IEDduZSG4QxoIK2olsaZIhrty4rv0u9/MLkmREuU6waXFARxDErmPeexj5rc79qLFIgpbYr7z/aiNdLi/r36R1n/dw4POjnvQOTxy2/uHh1juuvvu3g60v6NOOS2FZAnAThJF8r52f1b/f0qP4HrOEj6BCZ/6oS/9KBQwjRKQcw7JMoSPS77kAhoiWiYeB18eQ8hveaIqfQkTP+GeDO6aLSNt4nhRKHkoHR7e4sIyvAlY14NTY3BinQ36786dhItlIIVx2jt9fX5i2rbHvDm3kROkLVTBqOe226bxy+s+teG3LLDlXcwhnvuA759jO2QL9Tp6tdfBInyysZ93E0d+KFe1AROyFUuwlEDTOO0PBsRywiQTXILnJwnkr3bM5BwV6Q17DrVU7U+v1jp40dYep1ejIPJYYBoX/V+L3aZMzHF8/Y/VXS90df9X03h3Ylq//AtE4jloto+NeSu+Mw3jkRr2F9FSDfxLeOHh/mXhpNVqveyCuPFj+DTnusVLpzWJQg78sy+k2IUgmoGM0iohJ/jbwjIDOTaa8B8DQKkNWH5iuceA2kwllvpTeA/2FEwLazRPEz4c0woJgXvzCMz3VoN0hB8fv+4+/rn5Aa7f9H8BqzFGG9U0UNemeQwJl8skhPYxTH1kvbihSbdjzdvEkm38hr3BcJOh1s60fjLhZarfyjATOi+fuFpPbAho/NKbl8w43ibv6g11qJLHA8EVty09L3r9y82eXSyRzA/ADqEDa4rCF5AJPP0tfAr09wU8nF7bc+1Ou61k4lj9YXyf/f/R/b6+n+h+/+923KP93P8ftbEd/mLz2v//BfToH87YDx1crXN0Lr+Sswe3C68PoHHuto7G9rS5C6/34Uc43YeG5ALdmB9wQaXPsU2n1d4F/O5QwaHq9AwL3NZzLDgjLnstlx4P1eMBPe45Z8jrfL/l2vi1Rz2PsuCRBgAVdRwvAyeG3mwnRDoigQs0cejU1f5rKt9o2zZq3oUwshM+wzAj0J8C7cTAD7kgTwfWwNGGCc6SgNval1vvoBhidB0GDPLmadzABrhf46XUgaqKTVGAl9VM0d/DNgEYIe5jX2RSZD6hWIdBz1ahi5hbGCaVvgOo5FTqUWTFCxV6IHJWaHoVq3KPIq9pVpPbrHmRldVqFXvkM6hXWxfUahPL8cLXE6kWHjSWcRCxCUIWDGaEUgRPEI8IGN/BHENh07g+Qe6v9+1V13Tqrx09nKN02NIQO+MhT3ARjUjgaNULQy4qtHrXiCJlEU2nvuezIOW1ZfXgbFROb0mTTeVoIv5Xyimk8lVLu6xJpp1CViM9Xd+oXc6CAJseuMK6XVeiKNDIkULqIHRT4RTmGJ48AS+mCEvgylnTSDgkz/mh9buI7mnhRVmT7YIcBBbOhN864TIIDBoKG168eHp+dfEUYQhqfro/QtcVI65uyc9SwRDDX8QRuitibTA4UQ8tWsiNKOZhw8xR8z8ve6Pr897g8tzRP/aCoYe+x6bNYVUWmM2mMX6gpCorKxne8DuBPN/fqEPCDfghMMJgN+h0Jb01TNqMyMDcBXPBZeJ7ZvODIWj+T3A3Lxrs/c2H9123/QHQmY/zl5wfiWgacYIYvkHdkI+DzALUXVXhO4aAxEdf8MlH9OxP0BifkKuMYtttQ+NxqzN9/LhpwmNo4BkCfgAl3lnxQEtwuowtGI6cj9XwmNyczEKged4FJtG+8RL7YjCSCLN9TwD6IEDD8YjEMAbxBWoHjQn3fOXEft5rGmTqGZmqtgA5S7XyjmESIdrzMF7h6svqm9A7URvkmEDzR/WKDhXfVKes+vQKS7hgHnLIXDIGZ5sUEzayo3UqoygQTq71SFXSXrV6evttdCtx69jKoj/lZtsry/EZG+AaFjfAluiyUVTR62ywNgiZ54N82IXTSzxL5EFcDa+HRzqhpvz0VX+gxnNI4+Uv2Ixj3PhMcWORjucqQD2zrWEaMjdDsjW812OXGVSy9SI8lW5hW/S1/LMXLCccMcqUJzxE1HOPKMV0UxyuBBJWaYUOt98gTjPVE1CUV8Q4FSZuCSkPlLvGvErwNmOLKOrrDS4xL4kt4pvU3jIoeqCENT5VMoqmlcHS18lIraCdcKE8RxTgYzCOYY4HzBkb4xsLgo0t8dz2xn5iWxfZvlgbYqrFAtVoyhme3nELWhfpVtm+U8p8t4hMjX+oSDUyXzfL67LW13ZaTx5DD8BDdMFxzMpUx0/cn80luK32Q4cll5f5uTM8tSTRGA+j6OIEIdsJHmAUyk1j3SpozNhSiFEY+RgmBAvkKOZxjNFGl4+DZQKLiC7y9PPvMZ9BEH0aBUpLeiIj1AIZYayR6Zog/pfE34VOWrSaNTzEoWu2LrcdOFJtMWx1rUtCfuyW2x+XLPDl3eY4lLhtlVN1Gvl2OR4ZpGa/GF/OML5ME45jHHp3WE1QS430QMWTdqt9gF8ufjr42cPPwdoWwjOtHcTWYJivnxKkJea49gKCAzj0McatLjauOA2VGa1JmD9MwhynOBdRKWHFqDwOe3guR9QTs8nED2e7aDzyi6OA0ULahWngxzAc9vTY9Ib9K7VScMW7rc4Bfh0Aooa1oaEDPjK0VPtq5fEIP+MSG9kJSQLddlP1TVblnUz1HuroJQgBKyXFCY+TyMPdle5ssqhSTomPMrhPxoYYLuhAOvYJh7AEH7wlfiBgoffvSGwY37H1INpWv1olXQ1Wv0qP9d7a3hzp0ISQ69MPM3RJPLnTL+NIzjeU2cuhz8YUSEaooUKFvE8Zkh110XmEU4LdlBAgLdFfamCGqGvBkruCh91HDpnXyDHjOIqkkAmL8XChAWOG76pvOGwbGYqsjhUwwyoQ/lB8fZ69rjBmQZUK9dIT9wPV24QIJQ1TZbKaTQ1K1yaqML0x+QYV0iC6oUAWXPPxyBuu6UNXLyvBGBcC7slRmqTRstOXdG2sWKTT3VpM8OCbH4n1gbfqyk7lI/Ibu7/77rOmnR35t9//t/cOjw5W9/+dPXX/f3BY3///FVS+/x8miH+y3dqFT8yXChcO3V2d7js7ss+egfDwqB4iToBoqhPFCrqSR4JblvgslAKiEMScJRgT2urq/xlr7qIIzeb5Ct/msDrl5bbbN/ogL6ARck5txndw5rZXKeYtWYJXvWudT6YwuIgmPND3ju/8of3KUTlixTjksioB26e6t1y6bwjeY5zYlhLOgKek4cJhMI2LPJedmbMlqe2oPqOslWkMB73+2xPTIt3JsUp/ldrGF3UDhyNoa2F2FKomVC2wMFwubB5H3lzAATVKs8V047qCtQd4dEpT35cnGgg+Y5rf8SoFa12WXLYsLoWNXMunuY8H73iGoEqlhkvZeOGxgKPSZjEoECIBEXDscKiAIiKKLddvQxfofxHEnE+KeVNX39qSBDW3+oYyE+asTB/pa0fR8sRt8aqWFFeGULob4FFq1QtUHkj5l/SIdiOMYjOhU+o4BrQox8y7mSXREiHOMgzoiObjEi/+c4HOsudJdOJXmUVft8Gils7q3xa0VRnXD3SprfpZl7qpzumX0+862061eYo9TbBrZSh8ozom1j5QB90j1yBjoJEAivnTTL1Ofa9aN/KUgd4AKF1Di3zlpF3UftAp/bQ73alnxlO2nNAFKaXT+aVxuS8xf/72jBCWktKwGpWjSmxV96s38OWLfqSEfrNpQhOeGH+kC0gv8/bICzgLy0Xo19hyRteey1nmDDa9QOaG9Nampng4GdPx0iCnW2apj/q5a7H1Ox5Lt20gPQd44qGLa7qZneLqNwsgr1CV+eHcJelpyf3kNv9Y5ajgN6N4GRDimO4Wbiu6bumts7u6scir1Etnly4tsEz9dHbz2wssWj1n4qLs0FI5wGS6XkFr41G9ltYb3een9Ipa61FaW1Xc7l1leva3++EaQNdUU0011VRTTTXVVFNNNdVUU0011VRTTTXVVFNNNdVUU0011VRTTTXVVFNNNdVUU007/wUXynhXAFAAAA==' | base64 -d | tar xzf - -C $Q
ls -la $Q
pgrep -f "queue/q1.sh" >/dev/null && echo "q1 already running" || { nohup bash $Q/q1.sh >> $Q/q1.log 2>&1 & echo "q1 started"; }
pgrep -f "queue/t1.sh" >/dev/null && echo "t1 already running" || { nohup bash $Q/t1.sh >> $Q/t1.log 2>&1 & echo "t1 started"; }
sleep 20; tail -n 5 $Q/q1.log $Q/t1.log

total 17
drwx------ 2 root root 4096 Oct 24 10:35 .
drwx------ 8 root root 4096 Oct 24 10:35 ..
-rw------- 1 root root  938 Oct 24 10:34 common.sh
-rw------- 1 root root 5244 Oct 24 10:34 q1.sh
-rw------- 1 root root 1981 Oct 24 10:34 t1.sh
q1 started
t1 started
==> /content/drive/MyDrive/SEARLE/queue/q1.log <==
===== queue 1 start Sat Oct 24 10:35:52 AM UTC 2026
[10:35] START E1.7b-searle-circo

==> /content/drive/MyDrive/SEARLE/queue/t1.log <==
===== train queue 1 start Sat Oct 24 10:35:52 AM UTC 2026


In [30]:
%%bash
# THEO DÕI: chạy lại ô này bất cứ lúc nào
source /content/env.sh; Q=$DRIVE/queue; R=$DRIVE/results
date; echo "== q1"; tail -n 12 $Q/q1.log; echo "== t1"; tail -n 6 $Q/t1.log
for f in $(ls -t $R/*/stdout.log 2>/dev/null | head -1); do echo "== $f"; tail -c 300 $f | tr '\r' '\n' | grep -v '^$' | tail -2; done
for n in phi_scale_s0 phi_s0_clean phi_s0_degaug phi_s0_noise; do l=$R/E3.8a-train/$n.log; [ $n = phi_scale_s0 ] && l=$R/E1.14a-train-s0/train.log; [ -f $l ] && echo "$n: $(tail -c 200 $l | tr '\r' '\n' | grep -v '^$' | tail -1 | cut -c1-60)"; done
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

Sat Oct 24 11:03:42 AM UTC 2026
== q1
[10:58] OK    E1.8-sum-fiq-ex
[10:58] START E1.8-searle-circo-ex
[10:59] OK    E1.8-searle-circo-ex
[10:59] START E1.8-searle-fiq-ex
[11:00] OK    E1.8-searle-fiq-ex
[11:00] START E1.8-phiB32-circo-ex
[11:00] OK    E1.8-phiB32-circo-ex
[11:00] START E1.8-phiB32-fiq-ex
[11:01] OK    E1.8-phiB32-fiq-ex
[11:01] START E1.9-cbir-color-cirr
[11:02] OK    E1.9-cbir-color-cirr
[11:02] START E1.9-cbir-color-fiq-ex
== t1
===== train queue 1 start Sat Oct 24 10:35:52 AM UTC 2026
== /content/drive/MyDrive/SEARLE/results/E1.9-cbir-color-fiq-ex/stdout.log
fashioniq val gallery-dress:   1%|          | 1/120 [00:01<03:43,  1.88s/it]
phi_scale_s0: [90/100] total_loss: 4.735 gpt_loss: 0.142 distil_loss: 4.62
70 %, 19101 MiB


In [31]:
# TÓM TẮT: các lượt lỗi và số chính của mọi lượt đã xong
import json, glob, os, re
R = '/content/drive/MyDrive/SEARLE/results'
for q in ['q1', 'q2']:
    p = f'/content/drive/MyDrive/SEARLE/queue/{q}.log'
    if os.path.exists(p):
        print(q, 'FAIL:', [l.strip() for l in open(p) if 'FAIL' in l] or 'none')
KEYS = ['cirr_recall_at1', 'cirr_recall_at5', 'cirr_group_recall_at1', 'circo_map_at5', 'circo_map_at10', 'circo_map_at50', 'fiq_average_recall_at10', 'fiq_average_recall_at50']
rows = []
for f in sorted(glob.glob(R + '/*/metrics.json')):
    d = json.load(open(f)); mt = d.get('metrics', d)
    ks = [k for k in mt if k in KEYS] or [k for k in mt if re.search('average|map_at10|recall_at1$', k)]
    rows.append((os.path.basename(os.path.dirname(f)), ' '.join(f"{k.split('_',1)[1]}={mt[k]:.2f}" for k in ks if isinstance(mt[k], (int, float)))))
for n, s in rows: print(f"{n:38s} {s}")

q1 FAIL: none
q2 FAIL: none
E1.7a-searle-cirr                      recall_at1=25.66 recall_at5=55.47 group_recall_at1=55.08
E1.7b-searle-circo                     map_at5=6.82 map_at10=7.78 map_at50=9.76
E1.7c-searle-fiq                       fiq_recall_at10=22.85 fiq_recall_at50=42.02
E1.7d-phiB32-cirr                      recall_at1=25.47 recall_at5=55.23 group_recall_at1=55.20
E1.7e-phiB32-circo                     map_at5=6.42 map_at10=7.15 map_at50=9.08
E1.7f-phiB32-fiq                       fiq_recall_at10=22.59 fiq_recall_at50=41.52
E1.8-image-circo-ex                    map_at5=3.00 map_at10=3.34 map_at50=4.32
E1.8-image-cirr                        recall_at1=7.01 recall_at5=24.20 group_recall_at1=20.57
E1.8-image-fiq-ex                      fiq_recall_at10=6.13 fiq_recall_at50=13.15
E1.8-phiB32-circo-ex                   map_at5=8.78 map_at10=9.27 map_at50=11.23
E1.8-phiB32-fiq-ex                     fiq_recall_at10=22.74 fiq_recall_at50=41.57
E1.8-searle-circo-ex             

In [32]:
%%bash
# XEM NHANH: sửa danh sách đường dẫn rồi chạy lại
R=/content/drive/MyDrive/SEARLE/results
grep -E "C4|E2|H4|searle_c|check" /content/drive/MyDrive/SEARLE/queue/q1.log
for d in E2.0-stats-cirr E2.0-stats-fiq E2.0-stats-circo E2.2-agree-cirr E2.2-agree-fiq E2.2-agree-circo; do echo "=== $d"; ls $R/$d; f=$(ls $R/$d/*.md 2>/dev/null | head -1); [ -n "$f" ] && head -c 2500 $f; done

[10:45] START searle_cirr
[10:47] OK    searle_cirr
[10:47] START searle_circo
[10:47] OK    searle_circo
[10:47] START check_phiB32
[10:48] OK    check_phiB32
[10:48] C4: 4148 / 4148 queries with identical top-10 (100.00%)
[10:48] START E2.0-stats-cirr
[10:48] OK    E2.0-stats-cirr
[10:48] START E2.2-agree-cirr
[10:50] OK    E2.2-agree-cirr
[10:50] START E2.0-stats-fiq
[10:50] OK    E2.0-stats-fiq
[10:50] START E2.2-agree-fiq
[10:53] OK    E2.2-agree-fiq
[10:53] START E2.0-stats-circo
[10:53] OK    E2.0-stats-circo
[10:53] START E2.2-agree-circo
[10:54] OK    E2.2-agree-circo
=== E2.0-stats-cirr
report.md
stats.json
stdout.log
# Attribute statistics: cirr val

## E2.0 Share of queries per attribute

| Group | Queries | Any constraint | color | brightness | colorfulness | pattern | zoom | Negated | Relative |
|---|---|---|---|---|---|---|---|---|---|
| all | 4181 | 36.3% | 30.3% | 2.3% | 1.1% | 2.3% | 4.6% | 1.5% | 7.3% |

Most frequent constraints: color:white (405), color:black (223)

In [33]:
%%bash
# Hàng đợi 3 (đánh giá sàng lọc phi bền, tự chọn biến thể), train 2 (D10 phi_B32_robust + H16 cosine), hàng đợi 4 (đánh giá D11, E8.2a, E3.9)
source /content/env.sh; Q=$DRIVE/queue
echo 'H4sIAAAAAAAAA+0Ya3PaSDKf9Ss6hI2krIWQwE6CjWsd4lRcycVex5erKx9HyWIArSWNrJFwXMT/fbtnJBAYO9mrWqeuivkAmkc/pt89V62GmDz5e0cTx067Lf9xrP63d1ruE2fb3XnpOM22i+fwr+0+gebfzJccuci8FOBJynn20Lnv7f+fjmdP7Ysgti88MdGewe85yxm0OvD2NbCpF+ZeFvAY+AiyCa6D8FPG4iAew9RLAy/OBBiHrcYrz4Q8HrIUXsOQjVNvKOEE/Ar45VnC90ImERKqMweBnIbTNreQJCKOJXYvz3iEcD74Ex74rKSa8gvUUUkQjDQP5V48CVg0mOaDLPWCeNDjoXfRiIZbyLvZ0ATPU8RRf3t69OXQvqJ72T6PIh6jvWsfu/VTWzJuSWjtehIg1mScsgSsEdQUwJWLh2uwD/aQTe04D8NdGHIQIcNjO02axEwb8RROIIghmQQD0Rz4IfPmExSHl4/LWcwDwQhMA1Akn8I5Eax/tOsnDUIHfXj+HGYVXtjXxIq9CC9zssQLfPu2DB7yMfR34bbCpOOWXMofhqKFWpcGXCldA9l/BnUDNcXMmqaxqWEifXiGCoM9ZByQgX3Y43kGPh8y/ByF3lg0Go19vEbIUblw0q070OvW3V0Qk2CUgYtbaR4Dyrneg/oXsCzSv5XdJIykQfPFvXCGa5Y/Yf5lwoM4UxsktdATWSNBBnsHvfeHUKv/VsPTyEySZ9YwSBUF7fZ/VcNd8aNYZ6Akda7EAr/+8r7zyz/MPnz+cHRC/BrSbMgRRh6qcWjWdlE4cRbEOSK+RbzEzltiJyasSF0IRbfjFrOLME9x8kfCxvgX8utBGIwnGX4LL8wGCUsSRgciTs5UHqdzKRP45fOQpwMf5dNxi8sA6QzZU6ZdP7Hqs7d2x761/CBFSfWOTk9ReMpFyTkQpGIcUyUfclaUEkgftapLCo3C84PHfS6P94419PWedOwOXDB06IjUc/qbA3yKgYM8/VURPNgQcElQBNkCpUY65yEMR8BmYxsxoByWNK3VT/4NFuzt6YfH73T4BhljMPdyFVIa2ddMC6KEo8H/IXi8BVxop9DFvwaLp0HK43NdRgy9j7FLt1HQeZgJXUPGBJ471yuKdPUt0BeqlFNSpvyYq1POKgqV84pK56eRlvxeqNXV+0h4BKljJFswNDtSwyPkY6TPTm+Lq82SW2s2VCq2I5algS8adDtdHg8wUvKMbph42aTBvgYiE8bI7EDKsjyN4RPpnk5GiJjgMI54Q4MnLMZj5q5cjxpjlhl6gR7ZjHCjQBCd60QbL4B6Dwde5iDfKb8mgc2UXybkCOd6VV901SXnrCxI8ep9dV9lAN1CCjq5k47EMZhIjZTCkQ43JDqkq355dwUdCHlNwCPyH08RfGfus/I48Xye9BGrIcG2QOSRQQdNsCFEecjvxfJ5p92nrXZlqd1RS6ZEmaQYzIxRbZZ0nG1xW7Azk3+dhju6RUOdG730iFnBxbnTVwcMgSl3seyq5S1MuMsbLbVh1kwNszkjieGO0tuK4I3mlmmeN/savyQdJR2YSvGhIKckGwkYZCwSmApQiAk87cIyEvBi9FLEAftdkPQscsxbrUjeaDLeV4NfbsElu+mGXnQx9AAJ8Ut1N4kXybMQQVcMQVNbRAGPK4QEA3vqTiuGJEVVSFr/dHx22EGLn9cLF8zDKmUpJ/B4IfMg8sZM6KZWwPfeHx/1DlFGiqypYTSRoSXjPBRYQoQh87NBERgayQ3G02JSZCOVnSj0oE1EXnqDRclS2nb3nztrM7HMP/NE/Dj1X+b+9Pq/5TZ3FvV/a0fV/61N/f8YY7n+P6OqprBHF7sApyk9503LHRRluDEiE3aaTfRs/AWWcH8ipLsIFpdeZ2L2fO/szIF9LgIVdhMvxSTBwgZS+5cXoGti4OlAViGM7UGCuYRHWOGxAdEzZSwo3cQoXbtw0e8V+28OPh92a5blh0FiRVjBhqq6/BKcWW/slosOK6NAzLCgxHZF4D/lSsR3cHZgH9HeJ5Y5H/AgJbp1h4rKSnhTZpWlYe2eriJzVrsKKjnv9B6tB3qPRVlfjSNVIbqrdb0KbN26Ieno/y0C3boSCSNUhTXwri9Bn8n4CHX3FmOlqpmXOih7RWVriumVExAFAq1irMyMahTJMRtiKV0wS7meqmkFX6i9gwWl3K5pPuWdYobGhRl3KZGYcHr85p+fz0j5OLVGmArylKk4rWzF97DnsEv1K77U4QQNnjJaDXZ3F4hlZWJCBXGJVO4ggFMAvDChOkoAucuE52ualFvZaynd7WFPBGSc2GRRN4ecrm236Ah1XLhf7bmwVruky1kJ1PFfkxWQUhZObUmjUMyuaryVZFVXQ0hrZVW3C6NgTo4SHBKs4a7ERg5gk0SkNwlbgtqL7k3YCoL6NtnOlghUrkQyiPq+Duvs4PRMMQP1xWkDKAmL1Fe3GCB1lXwXXaT6JW+nFJwFiy2cUBjC5TiPLBWyZPSyLKzZAvVaYY1Shr4T+zewrdrMgjz8R1M63N+vypH0Q6mc7pfx3J+sCln1Y+tvefjpLW4qlo26cY+SCLVEcPyBHEl+vjs4+miaNTDhOTa9CBcjs8q6agpCmdJK3C689UJ5qzVflYDaMkgRrQnkVcMtHNwqVi0rTC2Buh7mIfZt89VrL43ypBTutnaNwf2B6PQzap3NuDuu2j+//mu9bC7qv/a2qv/am/rvMca69982GN4oo0eZqsualKid8jlWvgvSg658//WxKS4iQbFIgQMXe0fYUmLDRL15SjmKjr82v1Ox3VM3PfQau74iuirus1ILFU+c9z1d7j7myyXVN/MsvS5Ir0nbmnznW4nwUg8FjCUb3aXnuh8HWTzZ/SDMKLiiZPbu6Pfqu+djPnk+xOPSE2jxBrr6CPpXkMzFs/Yldf7qTXZuSZufC0u+wc6fEpRbDLzYC29EIJ8SlIsoFgursayIZRM+JO46KxxiNZXHorSaOa8vFKX5M8R6TjSs76p2dyfTf9fuCodXkGpyv8Xdc3hha8jOXckV6H5UZg8JbVlaTiklNXWtF8XX9kPik3JbyyXd48e4RLv5q1wi8nu48XmFHbQAP1i1sgtMWwI1mQz8omCmx7qwrO+cxsshBTAkXV5b0TdCAdbwjm1V4srcaKqqVziddinFSidpLl1iwezjvK61NxXnZmzGZmzGZmzGZmyGHH8Cm3fvBQAoAAA=' | base64 -d | tar xzf - -C $Q; ls $Q
for s in q3 t2 q4; do pgrep -f "queue/$s.sh" >/dev/null && echo "$s already running" || { nohup bash $Q/$s.sh >> $Q/$s.log 2>&1 & echo "$s started"; sleep 2; }; done

common.sh
q1.log
q1.sh
q2.log
q2.sh
q3.sh
q4.sh
t1.log
t1.sh
t2.sh
q3 started
t2 started
q4 started


In [29]:
%%bash
# Hàng đợi 2 (tự chờ hàng đợi 1 xong): E3.2, E2.3, E4.4, E4.5, E3.6, E3.7, phân tích H8, E3.4
source /content/env.sh; Q=$DRIVE/queue
echo 'H4sICIKK3GoAA3EyLnNoAK1Y227bOBB991fMpsYi6ZqWZSdN4GIX8DYtkodit0HRYp8EWqJtNpSokJQdt+i/75DUzbac5KEBHEMiOXPmzJV+9Vsw51kwp3rVewWfClYwGMOpNlQZDZsVy+DBvQyBZYk+m8K78BxiKvhcUcNlNoD3k+EYFvyRJaCYNrJ+fz6czPFrPJwANUbxeWEY6A1j+QB14fI5LApd7b2AXMk0N6hHs3QumJP8JsB/l5AywxQoOS+0yZjW8AdqXBaobwA3V0AzKrbaPuDu856WhYoZ9K/vbr+8Dxz+IJZpKrMhmrlZccEgXyqWA1nAiV9/CHHtBP6CIGHrICuEeAuJBC0QLrwZ2YeM9Vi8knDyp/0reRmD4wr6pwk17Oykh6YRQixN05onBmZlP4h3JUWiQS5wk2eLAS2MBJnBZDSCd7d3d2AU5RnwlC6ZhtNY5lu4Z8gM7rlWfM3Oeul9whWQvLIxlgmL4hXN8ERPFRn07yxxE9LyFPT//Q+MlEIHNa6oxBCpQjA9REWEeL3EKuhfzz7PAosp8JgI8TAJkYUBreJgV8A3LTNnWS6V8SIOgfTi/NjRDnM6tlUcX19NDwMP6MLGCgXB1kyQMSRsqWji105tsJ712GR8egY/oGFqTPo/wmAa/CT9MYm5QuBfkLCbW+g7lxDixWBUhS3X9cf4oOmakYcCDTRbz0xetI0/kP0WfvYWUsFHQEqXtNCa0wzmXKBHFBWQidQFH8L0y1EmuWbTMfQ/VoGIS5oKE+Usz5nCpZQlKKWRW2R6RVUOG84ymzsc+VQJsheJIt7uyZ+LQrXF11KeOZ1KS+vx4wmbCxnfN7bVJ7/lbNl5ZMW1YQ8QC4ops6RpStG7hmfssT4r5CYSfLky+4zYBXQNvi6tb9Gs6DbaSCUSrGocIz+nJl7VImMppIpiqndklnHmKtkUA2phAmX12txXHJOzyBJkZyF4Dp8/z+rMs/uJ3e52l/FUZ589vI0MfZSZTLc26aoYi5GkpXSSNeLjGAsYa2iLkxoOLxOSr/jfkypEvSry2j/6pDyiv7LlJhxNW+VYMaJods+z5QDLC4MFTbnYwiNsmDP01JbvM0fjh5kj0jEFcyfZlWL3YlEI94Co0c8ZfJfSxXAPwJ79ak+OhqML/BfiZ4yfSbkOdRqiKtJHNWTT/9qdhBY4Jp+FXD05xJYxe7J6WaLvfz3MxwMlxyEs+AM0ED7cfkJp7DEWRYKssQVTLMM28+tRoV4E5SKwHYY3YTgFHdu64/sm9r1kv2uix7Cx+o7q/Tbr4h4/FyX/TdCeEy+X0P6sm/9yvTJkdmhIhxRvRKPmglRgu5V4i4hmLas69OyJqZtCOJ62JwVstbb5NiHvZgkcFcoBwvWEN25yuPSEXddlua67ZbVqF552xdhjEuURivMFw8J/7Qq/jaSmANRYIpwdTNl23YGUZaaKlpTqez8blKFXd6BrpGr27uZ9K+M7Ve4S32xx0C3yMbFl0VXFXwtxp5w27bJVhJ+w4Bi8liWXpPJeA3nNte3B31nU6vi6XV6r8weFtAG+4/ZBy98DHwKDI5Y5Eux0dNVlWQ23TmWcXFwbgKoNDHDEY3mATdwM/OhH0PAE1ixGBTZKw2E4wjDFr9B/YX43FoUjUonyNr2g4ew1F4opR1Wdlcc6Tji8Ouw4hwCOY4vls9je/bOLrdQYS6y/T2mN5RG1rpQ/rdZGMByqxZNPKW3HpXUNsW60Xjzwgl2I7ErkrytcH/VEN+m6SMsX+xF2qPg4ph32j2Pa98Ala2GyMnZBec88D2zXQReUGHmPZbwkq53NLu53YPmtFWO1Ln/bw4nJBS8gyil+IoRqFzOaE7Z2g1QL1K7io5DQzhdDQsJ+HaQ9muZkiWdKjp6jCbce5WgXSVt7o6Jbs6XipZq7qHiBZjS6GrjxFj/1VRCqi9Va23sAzsbIXKsTnFc3r8Mp272O1jqqz/k2Vkf1ZBhWlay6qb3eqdu7wnutC7QUAu23l41CVO2xfChHlEYQpkhK1XaYJu2fF2D81+9h5w8Ktms3vyf8D7cWV9+iEQAA' | base64 -d | gunzip > $Q/q2.sh; bash -n $Q/q2.sh && echo syntax-ok
pgrep -f "queue/q2.sh" >/dev/null && echo "q2 already running" || { nohup bash $Q/q2.sh >> $Q/q2.log 2>&1 & echo "q2 started (waiting for q1)"; }

syntax-ok
q2 already running


---
## Mẫu ô lệnh

Chép ô này cho mỗi nhiệm vụ, dán lệnh vào cuối. Mỗi lượt đánh giá nên có `--cache-dir $DRIVE/cache` và
`--output-dir $DRIVE/results/<mã thí nghiệm>`; thêm `--note "..."` nếu cần ghi chú.

In [10]:
%%bash
source /content/env.sh
cd /content/XLA_SEARLE/SEARLE-main
CACHE="--cache-dir $DRIVE/cache_A100"
CIRR="--dataset cirr --dataset-path $DATA/CIRR"
nvidia-smi --query-gpu=name --format=csv,noheader
time $PY src/validate.py --eval-type searle $CIRR $CACHE --output-dir $DRIVE/results/E1.7a-searle-cirr
cat $DRIVE/results/E1.7a-searle-cirr/metrics.json

NVIDIA A100-SXM4-40GB
[run] python src/validate.py --eval-type searle --dataset cirr --dataset-path /content/data/CIRR --cache-dir /content/drive/MyDrive/SEARLE/cache_A100 --output-dir /content/drive/MyDrive/SEARLE/results/E1.7a-searle-cirr
[run] output dir /content/drive/MyDrive/SEARLE/results/E1.7a-searle-cirr, runner Chi, commit 924d9c4, gpu NVIDIA A100-SXM4-40GB
Target pad preprocess pipeline is used (ratio 1.25, bicubic)
Eval type = searle 	 exp name = None 	
CIRR val dataset in relative mode initialized
CIRR val dataset in classic mode initialized
extracting features: cirr val gallery
[cache] saved clip-cirr-val-gallery-ViT-B32-targetpad1.25-bicubic-a50e141f96.pt
CIRR val dataset in relative mode initialized
extracting features: cirr val references
[cache] saved clip-cirr-val-references-ViT-B32-targetpad1.25-bicubic-f7e1a2060d.pt
Extracting tokens using phi model
cirr_recall_at1 = 25.66
cirr_recall_at5 = 55.47
cirr_recall_at10 = 69.24
cirr_recall_at50 = 90.58
cirr_group_recall_at

Downloading: "https://github.com/miccunifi/SEARLE/archive/main.zip" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/miccunifi/SEARLE/releases/download/weights/SEARLE_ViT-B32.pt" to /root/.cache/torch/hub/checkpoints/SEARLE_ViT-B32.pt
100%|██████████| 24.0M/24.0M [00:00<00:00, 98.4MB/s]
100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 304MiB/s]
encoding queries: 100%|██████████| 131/131 [00:02<00:00, 61.91it/s]

real	1m41.769s
user	1m51.052s
sys	0m17.373s


In [1]:
%%bash
date; uptime; ls /content; ps aux | grep -E "queue/|train_phi|validate|precompute|tools/" | grep -v grep | awk '{print $2, $9, $12, $13, $14, $15, $16}' | head -20
D=/content/drive/MyDrive/SEARLE/queue; for f in q1 q2 q3 q4 t1 t2; do echo "== $f"; tail -n 4 $D/$f.log 2>&1; done
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

Sat Oct 24 01:15:12 PM UTC 2026
 13:15:13 up 32 min,  0 user,  load average: 0.16, 0.07, 0.01
sample_data
7 12:42 /datalab/web/app.js    
== q1
tail: cannot open '/content/drive/MyDrive/SEARLE/queue/q1.log' for reading: No such file or directory
== q2
tail: cannot open '/content/drive/MyDrive/SEARLE/queue/q2.log' for reading: No such file or directory
== q3
tail: cannot open '/content/drive/MyDrive/SEARLE/queue/q3.log' for reading: No such file or directory
== q4
tail: cannot open '/content/drive/MyDrive/SEARLE/queue/q4.log' for reading: No such file or directory
== t1
tail: cannot open '/content/drive/MyDrive/SEARLE/queue/t1.log' for reading: No such file or directory
== t2
tail: cannot open '/content/drive/MyDrive/SEARLE/queue/t2.log' for reading: No such file or directory
0 %, 0 MiB


## Chạy nền (train dài)

`nohup ... &` để lệnh chạy tiếp khi ô kết thúc; theo dõi bằng ô thứ hai. Colab ngắt thì chạy lại ô 0–5 rồi chạy đúng lệnh cũ, thêm `--resume`.

In [18]:
%%bash
source /content/env.sh
cd /content/XLA_SEARLE/SEARLE-main
BASE="--clip-model-name ViT-B/32 --imagenet-dataset-path $DATA/ImageNet1K --cirr-dataset-path $DATA/CIRR --save-training"
FCACHE="--features-cache-dir $DRIVE/cache/train_features"
ls data/oti_pseudo_tokens/imagenet/test/ | grep oti_B32_s0 || echo "WARNING: oti_B32_s0 not found"
mkdir -p $DRIVE/results/E1.14a-train-s0 $DRIVE/results/E3.8a-train
if [ -e data/phi_models/phi_scale_s0 ]; then echo "phi_scale_s0 exists: not started (add --resume)"; else nohup $PY src/train_phi.py --exp-name phi_scale_s0 $BASE --oti-exp-name oti_B32_s0 --train-on-oti-names --num-epochs 100 --validation-frequency 5 $FCACHE > $DRIVE/results/E1.14a-train-s0/train.log 2>&1 & fi
nohup $PY tools/precompute_degraded_features.py --imagenet-path $DATA/ImageNet1K --oti-exp-name oti_B32_s0 --degrade none,gauss_noise:1,gauss_noise:2,gauss_blur:1,gauss_blur:2,jpeg:1,jpeg:2,low_light:1,low_light:2 --out-dir $DRIVE/cache/imagenet_s0 > $DRIVE/results/E3.8a-train/precompute_s0.log 2>&1 &
sleep 90
echo "--- T1"; tail -n 5 $DRIVE/results/E1.14a-train-s0/train.log
echo "--- D7"; tail -n 5 $DRIVE/results/E3.8a-train/precompute_s0.log
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

oti_B32_s0
--- T1
filtering problematic concepts
100%|██████████| 20931/20931 [00:02<00:00, 8965.55it/s]
--train-on-oti-names: 33334/100000 images have an OTI token
extracting image features ImageNetDataset - test
 73%|███████▎  | 758/1042 [01:11<00:24, 11.45it/s]--- D7
33334 images x 9 versions
part 1/7:  35%|███▌      | 55/157 [01:16<01:11,  1.44it/s]utilization.gpu [%], memory.used [MiB]
58 %, 6807 MiB


In [ ]:
%%bash
source /content/env.sh
# tail -n 3 $DRIVE/results/<mã>/train.log
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## Xem kết quả đã có

In [ ]:
%%bash
source /content/env.sh
cd /content/XLA_SEARLE/SEARLE-main
$PY tools/collect_results.py --results-dir $DRIVE/results --out $DRIVE/results/summary.md > /dev/null
tail -n 40 $DRIVE/results/summary.md
echo; echo "--- 5 dòng cuối runs_$RUNNER.csv ---"
tail -n 5 $DRIVE/results/runs_$RUNNER.csv 2>/dev/null || echo "(chưa có lượt chạy nào)"

In [13]:
%%bash
source /content/env.sh
cd $DRIVE/outputs && pwd && ls -la
for d in GPT_phrases oti_pseudo_tokens oti_pseudo_tokens/imagenet oti_pseudo_tokens/imagenet/test similar_concepts phi_models; do echo "== $d"; ls -la $d 2>&1 | head -30; done
ls -la $DRIVE

/content/drive/MyDrive/SEARLE/outputs
total 24
drwx------ 6 root root 4096 Oct 24 09:24 .
drwx------ 7 root root 4096 Oct 24 09:24 ..
drwx------ 2 root root 4096 Oct 24 09:24 GPT_phrases
drwx------ 2 root root 4096 Oct 24 09:24 oti_pseudo_tokens
drwx------ 3 root root 4096 Oct 24 09:27 phi_models
drwx------ 2 root root 4096 Oct 24 09:24 similar_concepts
== GPT_phrases
total 8
drwx------ 2 root root 4096 Oct 24 09:24 .
drwx------ 6 root root 4096 Oct 24 09:24 ..
== oti_pseudo_tokens
total 8
drwx------ 2 root root 4096 Oct 24 09:24 .
drwx------ 6 root root 4096 Oct 24 09:24 ..
== oti_pseudo_tokens/imagenet
ls: cannot access 'oti_pseudo_tokens/imagenet': No such file or directory
== oti_pseudo_tokens/imagenet/test
ls: cannot access 'oti_pseudo_tokens/imagenet/test': No such file or directory
== similar_concepts
total 8
drwx------ 2 root root 4096 Oct 24 09:24 .
drwx------ 6 root root 4096 Oct 24 09:24 ..
== phi_models
total 12
drwx------ 3 root root 4096 Oct 24 09:27 .
drwx------ 6 root r

In [14]:
%%bash
OLD=/content/drive/MyDrive/outputs
ls -la $OLD/ && du -sh $OLD/GPT_phrases $OLD/similar_concepts $OLD/oti_pseudo_tokens $OLD/phi_models
find $OLD/GPT_phrases $OLD/similar_concepts $OLD/oti_pseudo_tokens $OLD/phi_models -maxdepth 3 | head -60

total 57
drwx------ 12 root root 4096 Oct  5 08:18 .
dr-x------  3 root root 4096 Oct  5 08:18 ..
drwx------  2 root root 4096 Oct 13 19:15 discarded
-rw-------  1 root root    0 Oct 15 15:25 eval3_all.done
-rw-------  1 root root    0 Oct 15 12:29 eval5_fiq.done
drwx------  2 root root 4096 Oct 13 18:52 GPT_phrases
drwx------  2 root root 4096 Oct 13 18:52 logs
-rw-------  1 root root  629 Oct 13 19:17 merge_oti.py
-rw-------  1 root root 1873 Oct 13 19:24 nvidia_smi.txt
-rw-------  1 root root    0 Oct 14 19:10 oti_B32.done
drwx------  3 root root 4096 Oct 13 18:52 oti_pseudo_tokens
-rw-------  1 root root    0 Oct 15 03:43 phi_B32.done
drwx------  2 root root 4096 Oct 13 18:52 phi_models
-rw-------  1 root root 1058 Oct 14 19:35 pip_freeze.txt
drwx------  2 root root 4096 Oct 13 18:59 prev_results_oct5
-rw-------  1 root root 2368 Oct 15 04:53 REPORT.md
-rw-------  1 root root 2235 Oct 15 19:09 REPORT_results.md
-rw-------  1 root root 1160 Oct 13 19:17 run_all2.sh
-rw-------  1 roo

In [15]:
%%bash
OLD=/content/drive/MyDrive/outputs
NEW=/content/drive/MyDrive/SEARLE/outputs
rmdir $NEW/phi_models/phi_scale_s0 2>/dev/null
time cp -rn $OLD/GPT_phrases $OLD/similar_concepts $OLD/oti_pseudo_tokens $OLD/phi_models $NEW/
du -sh $NEW/*
ls $NEW/oti_pseudo_tokens/imagenet/test/oti_B32_s0 $NEW/phi_models

699M	/content/drive/MyDrive/SEARLE/outputs/GPT_phrases
788M	/content/drive/MyDrive/SEARLE/outputs/oti_pseudo_tokens
505M	/content/drive/MyDrive/SEARLE/outputs/phi_models
318M	/content/drive/MyDrive/SEARLE/outputs/similar_concepts
/content/drive/MyDrive/SEARLE/outputs/oti_pseudo_tokens/imagenet/test/oti_B32_s0:
ema_oti_pseudo_tokens.pt
hyperparameters.json
image_names.pkl
oti_pseudo_tokens.pt
source_code.py

/content/drive/MyDrive/SEARLE/outputs/phi_models:
dryrun_2ep
dryrun_2ep (1)
dryrun_2ep (2)
dryrun_2ep (3)
phi_B32
phi_scale_s0


cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: cannot open '/content/drive/MyDrive/outputs/phi_models/phi_B32/train_metrics.gsheet' for reading: Operation not supported

real	1m33.155s
user	0m0.000s
sys	0m1.767s


In [16]:
!ls -laR /content/drive/MyDrive/SEARLE/outputs/phi_models/phi_scale_s0 | head; echo ---; ls -la /content/drive/MyDrive/outputs/phi_models/

/content/drive/MyDrive/SEARLE/outputs/phi_models/phi_scale_s0:
total 9
drwx------ 2 root root 4096 Oct 24 09:27 .
drwx------ 8 root root 4096 Oct 24 09:38 ..
-rw------- 1 root root 1019 Oct 24 09:27 hyperparameters.json
---
total 28
drwx------  7 root root 4096 Oct 13 18:52  .
drwx------ 12 root root 4096 Oct  5 08:18  ..
drwx------  2 root root 4096 Oct 21 11:25  dryrun_2ep
drwx------  2 root root 4096 Oct 21 11:21 'dryrun_2ep (1)'
drwx------  2 root root 4096 Oct 21 11:18 'dryrun_2ep (2)'
drwx------  2 root root 4096 Oct 21 11:11 'dryrun_2ep (3)'
drwx------  3 root root 4096 Oct 14 19:35  phi_B32


In [17]:
!rm -r /content/drive/MyDrive/SEARLE/outputs/phi_models/phi_scale_s0 && ls /content/drive/MyDrive/SEARLE/outputs/phi_models/

 dryrun_2ep  'dryrun_2ep (1)'  'dryrun_2ep (2)'  'dryrun_2ep (3)'   phi_B32
